# The MV Frontier and TIPS

In [1]:
import numpy as np
import pandas as pd

from utils import ANN, summary_stats, tangency_weights, portfolio_stats

pd.set_option('display.float_format', '{:.4f}'.format)

rets = pd.read_excel('./data/multi_asset_etf_data.xlsx', sheet_name='excess returns', index_col=0)
rets = rets.drop(columns='QAI')

mu = rets.mean()
sigma = rets.cov()
sharpe = summary_stats(rets)['Sharpe']

## 3. The MV Frontier

### 3.1 Tangency portfolio weights

$$w^{tan} = \frac{\Sigma^{-1}\tilde\mu}{\mathbf{1}'\Sigma^{-1}\tilde\mu}$$

- $w^{tan}$: $n \times 1$ vector of tangency portfolio weights
- $\tilde\mu$: $n \times 1$ vector of mean monthly excess returns (over SHV)
- $\Sigma$: $n \times n$ covariance matrix of monthly excess returns
- $\mathbf{1}$: $n \times 1$ vector of ones, so $\mathbf{1}'\Sigma^{-1}\tilde\mu$ makes the weights sum to 1
- $n$: number of assets (10)

In [2]:
w_tan = tangency_weights(mu, sigma)

tab = pd.DataFrame({'w_tan': w_tan, 'Sharpe': sharpe})
tab['Rank w'] = tab['w_tan'].rank(ascending=False).astype(int)
tab['Rank Sharpe'] = tab['Sharpe'].rank(ascending=False).astype(int)
tab.sort_values('w_tan', ascending=False)

,w_tan,Sharpe,Rank w,Rank Sharpe
IEF,1.3188,0.1384,1,8
SPY,1.2632,0.8979,2,1
HYG,0.2038,0.4705,3,2
EFA,0.1848,0.4291,4,3
DBC,0.0363,0.0755,5,9
EEM,-0.0010,0.2481,6,7
TIP,-0.1920,0.2568,7,6
IYR,-0.2237,0.4235,8,4
PSP,-0.4106,0.3617,9,5
BWX,-1.1797,-0.2096,10,10


In [3]:
print(f"Spearman rank correlation (weights vs Sharpe): {tab['w_tan'].corr(tab['Sharpe'], method='spearman'):.3f}")

Spearman rank correlation (weights vs Sharpe): 0.382


### 3.2 Does the ranking of weights align with the ranking of Sharpe ratios?

**No** (rank correlation ≈ 0.38). IEF has a low Sharpe (Rank 8) but the largest weight (Rank 1). IYR and PSP have high Sharpe but are shorted.

### 3.3 Tangency portfolio performance

In [4]:
perf_tan = portfolio_stats(w_tan, mu, sigma)
perf_tan.to_frame('Tangency')

,Tangency
Mean,0.1617
Vol,0.1096
Sharpe,1.4758


## 4. TIPS

The +0.0012 is applied to the monthly mean; $\Sigma$ is unchanged.

- $\Sigma$: covariance matrix of monthly excess returns

In [5]:
# (a) drop TIPS from the investment set
rets_ex = rets.drop(columns='TIP')
mu_ex, sigma_ex = rets_ex.mean(), rets_ex.cov()
w_ex = tangency_weights(mu_ex, sigma_ex)

# (b) raise the expected (monthly) excess return of TIPS by 0.0012
mu_up = mu.copy()
mu_up['TIP'] += 0.0012
w_up = tangency_weights(mu_up, sigma)

weights = pd.concat({
    'Base': w_tan,
    'No TIPS': w_ex,
    'TIPS +0.0012': w_up,
}, axis=1)
weights

,Base,No TIPS,TIPS +0.0012
BWX,-1.1797,-1.1375,-0.8602
DBC,0.0363,0.0258,-0.0433
EEM,-0.0010,-0.0062,-0.0402
EFA,0.1848,0.1881,0.2094
HYG,0.2038,0.1791,0.0172
IEF,1.3188,1.1616,0.1289
IYR,-0.2237,-0.2234,-0.2216
PSP,-0.4106,-0.4004,-0.3338
SPY,1.2632,1.2128,0.8819
TIP,-0.1920,NaN,1.2617


In [6]:
perf = pd.concat({
    'Base': perf_tan,
    'No TIPS': portfolio_stats(w_ex, mu_ex, sigma_ex),
    'TIPS +0.0012': portfolio_stats(w_up, mu_up, sigma),
}, axis=1).T
perf['Δ Sharpe vs Base'] = perf['Sharpe'] - perf.loc['Base', 'Sharpe']
perf

,Mean,Vol,Sharpe,Δ Sharpe vs Base
Base,0.1617,0.1096,1.4758,0.0000
No TIPS,0.1554,0.1054,1.4745,-0.0012
TIPS +0.0012,0.1327,0.0842,1.5749,0.0991


### How much does the tangency portfolio change?

- **Drop TIPS:** barely changes (Sharpe −0.001).
- **TIPS +0.0012:** Sharpe rises≈0.10. Weights shift a lot (TIP −0.19 → +1.26).

### Do TIPS warrant a separate asset class?

**No.** Dropping TIPS does not significantly change mean, vol and sharpe versus the base. But if the returns of TIPS significantly rises (+0.0012), TIPS should be considered a separate asset.

# 3. Allocations



In [7]:
mu_targ = 0.01
n = len(mu)

## Equally-weighted (EW)

In [8]:
w_ew = pd.Series(1 / n, index = mu.index)
mu_ew = w_ew @ mu

scaled_ew = mu_targ / mu_ew
w_ew_scale = w_ew * scaled_ew

#check
w_ew_scale @ mu

0.009999999999999998

## Risk Parity (RP)

In [9]:
w_rp = pd.Series(1 / np.diag(sigma), index = sigma.index)
w_rp_norm = w_rp / w_rp.sum()
mu_rp = w_rp_norm @ mu
scale_rp = mu_targ / mu_rp
w_rp_scale = w_rp_norm * scale_rp

# check
w_rp_scale @ mu

0.01

## Mean-Variance (MV)

In [ ]:
#w_tan is tangency portfolio for Mean-Variance optimization from earlier
w_mv_rescaled = w_tan * (mu_targ / (w_tan @ mu))
print("Sanity check: w_mv_rescaled @ mu = ", w_mv_rescaled @ mu)

#(w_tan @ mu) / mu_targ
#w_mv_rescaled = w_tan * w

Sanity check: w_mv_rescaled @ mu =  0.01


## Comparison


In [24]:
# Return
ret_ew = rets @ w_ew_scale
ret_rp = rets @ w_rp_scale
ret_mv = rets @ w_mv_rescaled

# Mean
mean_ew = ret_ew.mean() * 12
mean_rp = ret_rp.mean() * 12
mean_mv = ret_mv.mean() * 12


# Vol
vol_ew = ret_ew.std() * np.sqrt(12)
vol_rp = ret_rp.std() * np.sqrt(12)
vol_mv = ret_mv.std() * np.sqrt(12)

# Sharpe Ratio
sharpe_ew = mean_ew / vol_ew
sharpe_rp = mean_rp / vol_rp
sharpe_mv = mean_mv / vol_mv

labs = ["Mean", "Volatility", "Sharpe Ratio"]
comp = pd.concat({
    'Equally-Weighted': pd.Series([mean_ew, vol_ew, sharpe_ew], index=labs),
    'Risk-Parity': pd.Series([mean_rp, vol_rp, sharpe_rp], index=labs),
    'Mean-Variance': pd.Series([mean_mv, vol_mv, sharpe_mv], index=labs),
}, axis=1)
comp


,Equally-Weighted,Risk-Parity,Mean-Variance
Mean,0.1200,0.1200,0.1200
Volatility,0.2753,0.3261,0.0813
Sharpe Ratio,0.4359,0.3679,1.4758


The Mean-Variance portfolio performs best with a Sharpe ratio of 1.4758, compared to the lower Sharpe Ratios of .4359 and .3679 of the Equally-Weighted and Risk-Parity portfolios, respectively. Because the weights are normalized so that all returns are the same (weights don't sum to 1, all annualized mean returns are .12), this is fully explained by the Mean-Variance portfolio having a much lower volatility (.0813) compared to the volatilities of the Equally-Weighted (.2753 volatility) and Risk-Parity (.3261 volatility) portfolios.